# 04: Ranking and signals, from word counts to kernels

**Practice set:** `practice/04-ranking-retrieval.md` · **Exam tasks:** Jan 18, Section B2 (WonkaRank) and Section B3 (Pattern Pursuit: Convolution) · **Moves:** M1, M11, M12

## Hook

**Scene 1.** Your VLA hears *"move the motor to its home position"*. Somewhere in its memory is a library of 1,000 stored skills, each with a short text description: "open the gripper", "spin the motor", "move the arm to the cube", and so on. It can't try all 1,000. It has to **fetch the right few first**. How does it turn a sentence into a score for each skill, so it can sort them? And once it hands you a sorted list, how do you grade the list itself? Was the right skill at the top, or buried at position 7?

**Scene 2.** The IMU on your arm streams acceleration readings 100 times a second. The stream is noisy. Somewhere in it the arm **bumped** a table leg, and somewhere else it **jerked** when a motor stalled. You can't stare at 6,000 numbers a minute. A *kernel* of three numbers, slid along the stream, can flag both for you, and you can design that kernel from a list of requirements.

Both scenes share one move: **turn a messy collection into numbers you can sort or threshold.** Part 1 does it for documents (ranking). Part 2 does it for signals (convolution).

## Before you start

- **You need:** fractions, and precision and recall from notebook 01, section 2. Logarithms are explained here from scratch.
- **How to use this notebook:** run the cells in order. When you see **PREDICT**, write your guess *before* running the next cell. When you see **BUILD**, write the function yourself; the test cell tells you when it's right. Hints sit in collapsed blocks. Open one only after a real attempt.
- **Pen and paper stay on the desk.** The exam has no computer. The notebook is for checking and exploring *after* you've reasoned it out.

In [ ]:
import math
import prep
import numpy as np
import matplotlib.pyplot as plt

---
# Part 1: Ranking

## 1. Term frequency: count the words

**Idea.** Treat each skill description as a bag of words. A *query* is the sentence you're searching with. The simplest way to say "this description matches the query" is: **it uses the query's words a lot.** So count them.

- A **term** is a word.
- A **document** is one item you might return (here: one skill description).
- The **term frequency** tf(t, d) is the number of times term t appears in document d. Example: in "grasp the cube then grasp the ball", tf(grasp) = 2 and tf(the) = 2.

**Smallest example.** A library of four skills, and the query "move the motor".

In [ ]:
library = {
    "K1": "move the arm to the cube",
    "K2": "spin the motor",
    "K3": "move the motor to home",
    "K4": "open the gripper",
}
query = "move the motor"
qwords = query.split()

rows = []
for name, text in library.items():
    words = text.split()
    rows.append([name, text] + [words.count(w) for w in qwords])
prep.table(rows, header=["skill", "text"] + [f"tf({w})" for w in qwords])

**PREDICT:** score each skill by *adding up* its three tf counts. Which skill(s) come out on top? Is that the skill you'd want the robot to run?

In [ ]:
for row in rows:
    print(f"{row[0]}: raw count total = {sum(row[2:])}")

K1 ties with K3, even though K1 has nothing to do with motors. It scores because it says "the" twice. A word like "the" appears everywhere, so it shouldn't count as evidence of anything. That is the problem the next idea fixes.

---
## 2. Rare words matter: document frequency and idf

**Idea.** A word is good evidence when it is **rare** across the library. If every skill says "the", seeing "the" tells you nothing. If only one skill says "spin", seeing "spin" points straight at it.

- The **document frequency** df(t) is the number of *documents* that contain t at least once. (Documents, not occurrences: K1 says "the" twice but adds only 1 to df(the).)
- The **inverse document frequency** is idf(t) = log₂(N / df(t)), where N is the number of documents.

**What is log₂?** log₂(x) is the power you raise 2 to in order to get x.
- log₂ 8 = 3 because 2³ = 8. log₂ 2 = 1. log₂ 1 = 0 because 2⁰ = 1.
- It also works between powers of 2: log₂ 3 ≈ 1.585, because 2^1.585 ≈ 3. It sits between log₂ 2 = 1 and log₂ 4 = 2, as it should.
- Dividing inside the log subtracts outside: log₂(a/b) = log₂ a − log₂ b. (Dividing powers of 2 subtracts exponents: 2⁵ ÷ 2² = 2³.)

In [ ]:
rows = [[x, math.log2(x), 2 ** math.log2(x)] for x in [1, 2, 4, 8, 3, 4 / 3, 5 / 4]]
prep.table(rows, header=["x", "log2(x)", "check: 2**log2(x)"])

**Why a log and not just N/df?** Halving how many documents contain a word adds exactly **1** to its idf (log₂(2·x) = 1 + log₂ x). So "twice as rare" earns a fixed bonus instead of doubling the weight. That stops one ultra-rare word from swamping everything else.

**PREDICT:** in our 4-skill library, what is idf("the")? What is idf("spin")? Work them out from the definition before running.

In [ ]:
N = len(library)
vocab = ["move", "the", "motor", "spin", "arm"]
rows = []
for w in vocab:
    df = sum(w in text.split() for text in library.values())
    rows.append([w, df, f"log2({N}/{df})", math.log2(N / df)])
prep.table(rows, header=["term", "df", "idf formula", "idf"])

A word in every document gets idf = log₂(N/N) = log₂ 1 = **0**, so it adds nothing however often it appears. A word in a single document gets the biggest weight.

**Putting it together.** The tf-idf score of a document for a query adds one contribution per query word:

$$\text{Score}(d) = \sum_{t \in \text{query}} \text{tf}(t, d) \cdot \text{idf}(t)$$

"Use the query words often (tf) **and** use rare ones (idf)." Here is every intermediate number for "move the motor":

In [ ]:
idf = {w: math.log2(N / sum(w in t.split() for t in library.values())) for w in qwords}
rows = []
for name, text in library.items():
    words = text.split()
    parts = [words.count(w) * idf[w] for w in qwords]
    rows.append([name] + [f"{words.count(w)} x {idf[w]:.2f}" for w in qwords] + [sum(parts)])
prep.table(rows, header=["skill"] + [f"tf x idf ({w})" for w in qwords] + ["score"])

K3 now wins clearly, and K1 only gets credit for "move". **Aha: rare words carry the information.** The idf weight is how a ranker learns to ignore "the" without anyone telling it which words are boring.

### BUILD 1: `tf_idf_scores`
`docs` is a dict {name: text}. `query` is a string. Return a dict {name: score}, using exactly the formula above (raw counts for tf, log₂ for idf).
- Treat the query as a set of distinct words (count each query word once).
- Skip any query word that appears in **no** document (its df is 0, so its idf is undefined; it can't help any document anyway).

In [ ]:
def tf_idf_scores(docs, query):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(tf_idf_scores, [
    ((library, "move the motor"), {"K1": 1.0, "K2": 1.0, "K3": 2.0, "K4": 0.0}),
    (({"a": "x y", "b": "y"}, "x"), {"a": 1.0, "b": 0.0}),
    (({"a": "x x", "b": "y"}, "x y"), {"a": 2.0, "b": 1.0}),
    (({"a": "x", "b": "y"}, "z x"), {"a": 1.0, "b": 0.0}),
])

### EXPLORE
**PREDICT 1:** for the query "spin the motor", which skill wins, and by roughly how much?

**PREDICT 2:** add a fifth skill, "motor test", to the library. Which idf values change, and in which direction? Does K1's score for "move the motor" go up, down or stay the same? (Careful: does "motor test" contain "the"?)

In [ ]:
if prep.ready(tf_idf_scores, {"a": "x"}, "x"):
    print("spin the motor:", {k: round(v, 3) for k, v in tf_idf_scores(library, "spin the motor").items()})
    bigger = dict(library, K5="motor test")
    print("5 skills, move the motor:", {k: round(v, 3) for k, v in tf_idf_scores(bigger, "move the motor").items()})

With five skills, "the" is no longer in *every* document, so its idf rises above 0 and K1's two "the"s start to count again. idf is a property of the **whole library**, not of one document. Add or remove a document and every score can shift.

**Now do practice problem 4.1 on paper**, then check:

In [ ]:
prep.check("4.1a", ...)   # list of document names, best first, e.g. ["Dw", "Dx", "Dy", "Dz"]
prep.check("4.1b", ...)   # the idf of a word that appears in every document

---
## 3. PageRank: where does a random surfer spend their time?

**Idea.** Sometimes documents point at each other: web pages link to pages, papers cite papers, and a robot skill like "tidy the desk" calls sub-skills like "grasp" and "place". A page is *important* if **important pages point to it**. That sounds circular, and it is. A random surfer breaks the circle:

- The surfer sits on some page. Each tick, they flip a biased coin.
- With probability **d** (the *damping factor*, usually 0.85) they click one of the page's outgoing links, chosen at random.
- With probability **1 − d** they get bored and **teleport** to a page chosen at random from all N pages.

**PageRank** PR(p) is the long-run fraction of ticks the surfer spends on page p.

**Smallest example.** Five pages. Read the table both ways: "links out to" is how the graph is given; "linked from" is what you'll need later.

In [ ]:
web = {"A": ["B"], "B": ["C"], "C": ["A", "B"], "D": ["B"], "E": ["A", "D"]}

rows = []
for p in web:
    senders = [q for q in web if p in web[q]]
    rows.append([p, ", ".join(web[p]), ", ".join(senders) or "nobody"])
prep.table(rows, header=["page", "links out to", "linked from"])

**PREDICT:** which page will the surfer visit most often? Which least? Write your ranking of all five before running.

First, watch the surfer take 12 steps (d = 0.85; "coin" is a random number between 0 and 1, and the surfer follows a link when coin < 0.85):

In [ ]:
rng = np.random.default_rng(0)
d = 0.85
pages = list(web)
page = "A"
rows = []
for step in range(1, 13):
    coin = rng.random()
    if coin < d:
        nxt, how = web[page][rng.integers(len(web[page]))], "follow a link"
    else:
        nxt, how = pages[rng.integers(len(pages))], "teleport"
    rows.append([step, page, round(coin, 2), how, nxt])
    page = nxt
prep.table(rows, header=["step", "on page", "coin", "action", "next page"])

Now let the surfer run for 100,000 steps and count where they were:

In [ ]:
rng = np.random.default_rng(1)
page, visits = "A", {p: 0 for p in web}
for _ in range(100_000):
    visits[page] += 1
    if rng.random() < d:
        page = web[page][rng.integers(len(web[page]))]
    else:
        page = pages[rng.integers(len(pages))]
sim = {p: visits[p] / 100_000 for p in web}
prep.table([[p, sim[p]] for p in web], header=["page", "fraction of time"])

How did your ranking do? Notice E: nobody links to E, so the surfer only lands there by teleporting. That happens with probability (1 − d) × (1/N) = 0.15 × 1/5 = 0.03 per tick, which matches the simulation.

### From simulation to formula
Simulating is slow and noisy. Instead, track the **probability** of being on each page.

Where can the surfer be at tick t+1 on page p? Exactly two ways:
1. They **teleported** (probability 1 − d) and the random pick was p (probability 1/N). Total: (1 − d)/N.
2. They were on some page q that links to p (probability PR(q)), they **followed a link** (probability d), and they picked the link to p out of q's out(q) links (probability 1/out(q)). Here out(q) is the number of outgoing links from q.

Add the two ways:

$$PR_{\text{new}}(p) = \frac{1-d}{N} + d \sum_{q \to p} \frac{PR_{\text{old}}(q)}{\text{out}(q)}$$

Start with PR = 1/N everywhere, and update **all pages simultaneously**: every new value uses only old values. Here is one update, every term shown:

In [ ]:
N = len(web)
pr0 = {p: 1 / N for p in web}
rows = []
for p in web:
    senders = [q for q in web if p in web[q]]
    inflow = sum(pr0[q] / len(web[q]) for q in senders)
    terms = " + ".join(f"{pr0[q]:.2f}/{len(web[q])}" for q in senders) or "(nobody)"
    rows.append([p, (1 - d) / N, terms, inflow, (1 - d) / N + d * inflow])
prep.table(rows, header=["page", "(1-d)/N", "sum of PR(q)/out(q)", "= in-flow", "new PR"])

**PREDICT:** add up the five "new PR" values. What do you get, and why must it be that? (Think about what each page does with its own PR.)

<details><summary>Check your reasoning</summary>

Each page q splits all of its PR evenly across its out-links, so the d-part hands out d × (total PR) = d in total. The teleport part hands out (1 − d)/N to each of N pages, which is 1 − d in total. Together: d + (1 − d) = 1. The total stays 1 **as long as every page has at least one out-link**. Remember that condition; it matters below.

</details>

### BUILD 2: `pagerank`
`links` is a dict {page: list of pages it links to}; every page appears as a key. Start with PR = 1/N for every page, apply the update `iters` times (simultaneously each time), and return a dict {page: PR}.

In [ ]:
def pagerank(links, d, iters):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(pagerank, [
    (({"A": ["B"], "B": ["A"]}, 0.85, 1), {"A": 0.5, "B": 0.5}),
    (({"A": ["B"], "B": ["A"], "C": ["A"]}, 0.5, 1), {"A": 1 / 2, "B": 1 / 3, "C": 1 / 6}),
    (({"A": ["B", "C"], "B": ["A"], "C": ["A"]}, 1.0, 1), {"A": 2 / 3, "B": 1 / 6, "C": 1 / 6}),
    (({"A": ["B"], "B": ["A"], "C": ["A"]}, 0.0, 5), {"A": 1 / 3, "B": 1 / 3, "C": 1 / 3}),
])

### EXPLORE: iterate to convergence
**PREDICT:** after one update, B leads (see the table above). Will B still lead after 50 updates? Will the iterated formula agree with the 100,000-step simulation?

A sequence of updates **converges** when the values stop changing (to the decimals you care about).

In [ ]:
if prep.ready(pagerank, {"A": ["A"]}, 0.85, 1):
    its = list(range(0, 31))
    hist = {p: [pagerank(web, 0.85, t)[p] for t in its] for p in web}
    plt.figure(figsize=(6, 3.5))
    for p in web:
        plt.plot(its, hist[p], marker=".", label=p)
    plt.xlabel("number of updates"); plt.ylabel("PR"); plt.legend(); plt.grid(alpha=.3)
    plt.title("PageRank settles down"); plt.show()
    final = pagerank(web, 0.85, 200)
    prep.table([[p, final[p], sim[p]] for p in web], header=["page", "PR after 200 updates", "simulation"])

Two things to notice:
- **One update is not convergence.** C overtakes B for a few rounds, then B wins in the end. If a question asks "at convergence", iterate (or solve the equations); don't stop after one step.
- **The formula and the surfer agree.** The update rule *is* the surfer, written as probabilities. The values it settles on are a **fixed point**: plugging them into the update gives the same values back.

**PREDICT:** what is PR(E) after 1 update? After 2? After 50? Explain in one sentence why it can never change after the first update.

In [ ]:
if prep.ready(pagerank, {"A": ["A"]}, 0.85, 1):
    for t in [1, 2, 50]:
        print(f"after {t:>2} updates: PR(E) = {pagerank(web, 0.85, t)['E']:.4f}")

### EXPLORE: a page with no way out
A **dangling** page has no outgoing links. Add page F that nobody can leave, and let A link to it.

**PREDICT:** after 50 updates, what happens to the *total* PR? (Look back at the "Check your reasoning" block.)

In [ ]:
if prep.ready(pagerank, {"A": ["A"]}, 0.85, 1):
    leaky = dict(web, A=["B", "F"], F=[])
    for t in [0, 1, 5, 50]:
        print(f"after {t:>2} updates: total PR = {sum(pagerank(leaky, 0.85, t).values()):.4f}")

Rank flows into F and has nowhere to go, so it vanishes. Real PageRank fixes this by pretending a dangling page links to **every** page. The exam's formula (and problem 4.2) has no dangling pages, so the plain formula is all you need there. As always, use the definition the question gives you (move **M1**).

**Now do practice problem 4.2 on paper**, then check:

In [ ]:
prep.check("4.2a", ...)     # floor(1000 x PR(C)) after one update
prep.check("4.2b", ...)     # letter of the page with the highest PR at convergence
prep.check("4.2b_D", ...)   # floor(1000 x PR(D)) at convergence

---
## 4. Precision@k, recall@k and F1@k

**Idea.** A ranker returns a sorted list. The user only looks at the first k items. So grade the **top k** with the metrics from notebook 01:
- **Precision@k** = (relevant items in the top k) ÷ k. "Of what I showed, how much was useful?"
- **Recall@k** = (relevant items in the top k) ÷ R, where R is the number of relevant items **in the whole collection**. "Of everything useful, how much did I show?"

Write hits(k) for the number of relevant items in the top k.

**Smallest example.** Your VLA retrieves 8 skills for a command. The relevant ones are s2, s3, s6 and s9. Notice that s9 never made it into the list.

In [ ]:
ranking = ["s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8"]
relevant = {"s2", "s3", "s6", "s9"}
R = len(relevant)

rows, hits = [], 0
for k, doc in enumerate(ranking, start=1):
    hit = doc in relevant
    hits += hit
    rows.append([k, doc, "hit" if hit else "-", hits, f"{hits}/{k}", f"{hits}/{R}"])
prep.table(rows, header=["k", "item at k", "relevant?", "hits(k)", "P@k", "R@k"])

**PREDICT:** as k grows from 1 to 8, can recall@k ever go *down*? Can precision@k ever go *up*? What has to happen at position k for each?

<details><summary>Check your reasoning</summary>

Recall's denominator R is fixed, and hits(k) never decreases, so recall never goes down. It goes up exactly at a hit. Precision's denominator grows by 1 every step, so it falls at a miss and rises at a hit (unless it's already 1). Here recall can never reach 1, because s9 isn't in the list at all.

</details>

### BUILD 3: `precision_recall_at_k`
`ranking` is a list of item names, best first. `relevant` is a set of all relevant names (some may be missing from the ranking). Return the tuple `(precision_at_k, recall_at_k)`.

In [ ]:
def precision_recall_at_k(ranking, relevant, k):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(precision_recall_at_k, [
    ((["a", "b", "c", "d"], {"b", "d", "z"}, 2), (0.5, 1 / 3)),
    ((["a", "b", "c", "d"], {"b", "d", "z"}, 4), (0.5, 2 / 3)),
    ((["a", "b", "c", "d"], {"b", "d", "z"}, 1), (0.0, 0.0)),
    ((["a"], {"a"}, 1), (1.0, 1.0)),
])

### F1@k, and a shortcut you can derive
F1 combines the two (notebook 01): F1 = 2·P·R ÷ (P + R). It's high only if both are high.

**Your turn to derive.** Write P@k = h/k and R@k = h/R, where h = hits(k). Substitute into 2PR/(P+R) and simplify as far as you can, on paper.

<details><summary>Hint 1</summary>

The numerator is 2 × (h/k) × (h/R) = 2h²/(kR). Put the denominator h/k + h/R over the common denominator kR.

</details>

<details><summary>Hint 2</summary>

The denominator is h(R + k)/(kR). Divide: the kR cancels, and so does one factor of h.

</details>

Run the cell below only after you have a formula. It prints F1 computed the long way next to the short formula you should have found.

In [ ]:
if prep.ready(precision_recall_at_k, ["a"], {"a"}, 1):
    rows = []
    for k in range(1, len(ranking) + 1):
        p, r = precision_recall_at_k(ranking, relevant, k)
        h = round(p * k)
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        rows.append([k, h, f1, 2 * h / (k + R)])
    prep.table(rows, header=["k", "hits", "F1 = 2PR/(P+R)", "2*hits/(k+R)"])

**F1@k = 2·hits(k) / (k + R).** Now read it:
- A **miss** at position k+1 keeps hits the same and makes the bottom bigger, so F1 **falls**.
- A **hit** raises the top by 2 and the bottom by 1. Does F1 always rise then? Prove it: compare 2(h+1)/(k+1+R) with 2h/(k+R) by cross-multiplying. (You'll need one fact: h ≤ k.)

<details><summary>Check your proof</summary>

Both bottoms are positive, so 2(h+1)/(k+1+R) > 2h/(k+R) exactly when (h+1)(k+R) > h(k+1+R). Expand: hk + hR + k + R > hk + h + hR, which simplifies to k + R > h. That is always true, because h ≤ k and R ≥ 1.

</details>

**Aha: F1 rises at every hit and falls at every miss.** So the best cut-off k is always right after some hit. You only need to check those positions. In the example above, F1 peaks at k = 6, not at the first little peak (k = 3). Checking every hit position is what catches that.

**Now do practice problem 4.4 on paper**, then check:

In [ ]:
prep.check("4.4", ...)   # the k that maximises F1@k

---
## 5. DCG: graded relevance and the position discount

**Idea.** Precision@k has two blind spots:
- It treats every relevant item the same. A perfect match and a so-so match both count as 1.
- It ignores order inside the top k. A hit at position 1 and a hit at position k count the same.

**Discounted cumulative gain** fixes both. Each item has a **graded relevance** rel (0 = useless, bigger = better), and the item at position i is multiplied by a **discount** 1/log₂(i + 1):

$$\text{DCG@}k = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)}$$

The discount table is worth knowing by heart for the first few positions:

In [ ]:
rows = [[i, math.log2(i + 1), 1 / math.log2(i + 1)] for i in range(1, 11)]
prep.table(rows, header=["position i", "log2(i+1)", "discount 1/log2(i+1)"])

Position 1 counts fully. Position 3 counts half (log₂ 4 = 2). Position 7 counts a third (log₂ 8 = 3). Notice the **gaps** between neighbouring discounts: 1 → 0.631 is a drop of 0.369, but 0.356 → 0.333 (positions 6 → 7) is a drop of only 0.023. The discount falls fast at the top and slowly further down.

**Smallest example.** Relevances [2, 0, 3] in positions 1–3.

In [ ]:
rels = [2, 0, 3]
rows, running = [], 0
for i, rel in enumerate(rels, start=1):
    disc = 1 / math.log2(i + 1)
    running += rel * disc
    rows.append([i, rel, disc, rel * disc, running])
prep.table(rows, header=["position", "rel", "discount", "rel x discount", "running DCG"])

**PREDICT:** which order of these same three items gives the *largest* DCG? Why must it be that order?

That best order is the **ideal** ranking, and its DCG is the most any order could score. **Normalised DCG** compares a ranking with its ideal: nDCG = DCG ÷ DCG(ideal order), a number between 0 and 1.

### BUILD 4: `dcg`
Return DCG of a list of relevances, with position 1 first. (DCG@k is just `dcg(rels[:k])`.)

In [ ]:
def dcg(rels):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(dcg, [
    (([1],), 1.0),
    (([0, 1],), 1 / math.log2(3)),
    (([3, 2, 0],), 3 + 2 / math.log2(3)),
    (([1, 1, 1],), 1 + 1 / math.log2(3) + 0.5),
    (([],), 0),
])

In [ ]:
def ndcg(rels):
    return dcg(rels) / dcg(sorted(rels, reverse=True))

if prep.ready(dcg, [1]):
    print(f"DCG {dcg(rels):.3f}, ideal {dcg(sorted(rels, reverse=True)):.3f}, nDCG {ndcg(rels):.3f}")

### Which single swap helps most?
You may swap **one adjacent pair** of results, at positions i and i+1. Say they hold relevances a and b, with discounts D_i and D_{i+1}.

**Derive it on paper first.** Only those two terms of the sum change. Write the "before" and "after" contributions and subtract.

<details><summary>Hint</summary>

Before: a·D_i + b·D_{i+1}. After: b·D_i + a·D_{i+1}. Subtract and factor. You should get a product of two differences.

</details>

**PREDICT:** for relevances [1, 3, 2, 0, 2], which adjacent swap raises DCG the most? Which ones make it worse?

In [ ]:
if prep.ready(dcg, [1]):
    r = [1, 3, 2, 0, 2]
    disc = [1 / math.log2(i + 1) for i in range(1, len(r) + 1)]
    rows = []
    for i in range(len(r) - 1):
        s = r.copy(); s[i], s[i + 1] = s[i + 1], s[i]
        d_rel, d_disc = r[i + 1] - r[i], disc[i] - disc[i + 1]
        rows.append([f"{i + 1}<->{i + 2}", dcg(s) - dcg(r), d_rel, d_disc, d_rel * d_disc])
    prep.table(rows, header=["swap", "change in DCG", "rel(i+1) - rel(i)", "D_i - D_(i+1)", "product"])

**Aha: swap gain = Δrel × Δdiscount.** A swap helps only if it moves the *more* relevant item up (Δrel > 0). And the same Δrel is worth much more near the top, because that's where the discount drops fastest (move **M12**).

**EXPLORE. PREDICT:** in [2, 3, 0, 0, 0, 0, 5], is it better to swap positions 1↔2 (Δrel = 1) or positions 6↔7 (Δrel = 5)? Use the discount table to estimate both gains before running.

In [ ]:
if prep.ready(dcg, [1]):
    r = [2, 3, 0, 0, 0, 0, 5]
    for i in [0, 5]:
        s = r.copy(); s[i], s[i + 1] = s[i + 1], s[i]
        print(f"swap {i + 1}<->{i + 2}: DCG changes by {dcg(s) - dcg(r):+.3f}")

**Now do practice problem 4.3 on paper**, then check:

In [ ]:
prep.check("4.3a", ...)   # floor(100 x nDCG@5)
prep.check("4.3b", ...)   # the two positions you swap, as a list [i, i+1]

---
## 6. Exam walkthrough: Jan 18, Section B2 (WonkaRank)

> Do this **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`, pages 15–18.

Read the background box on page 15 slowly. It defines Recall@k, Precision@k, DCG@k and a new metric, PI-Recall@k. Use **those** definitions (move **M1**). Here is the chocolate table as data, so you don't have to retype it:

In [ ]:
# ingredient vector (Cocoa, Hazelnut, Caramel, Mint) and relevance, copied from page 16
ingredients = {"D1": (1, 1, 0, 0), "D2": (1, 1, 1, 0), "D3": (1, 0, 1, 0), "D4": (0, 1, 0, 0), "D5": (1, 1, 0, 1),
               "D6": (0, 0, 1, 1), "D7": (1, 0, 0, 0), "D8": (1, 1, 0, 0), "D9": (0, 1, 1, 0)}
relevance = {"D1": 10, "D2": 0, "D3": 0, "D4": 0, "D5": 3, "D6": 0, "D7": 0, "D8": 5, "D9": 3}
relevant_set = {c for c in relevance if relevance[c] > 0}


def rank_by(score):
    """Sort chocolate names by score, highest first; ties go to the smaller ID number (the paper's rule)."""
    return sorted(score, key=lambda c: (-score[c], int(c[1:])))

### Step 1 (Q1): Recall@5
Charlie buys [D2, D3, D5, D6, D7, D8] in that order. What is ⌊100 × Recall@5⌋? Do it on paper, then use the cell below if you want to confirm with your BUILD 3 function.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

Recall's denominator R counts the relevant chocolates in the **whole collection**, not just the ones Charlie bought.

</details>

<details><summary>Hint 2</summary>

"@5" means only the first five purchases count. The sixth one is ignored, even if it's relevant.

</details>

In [ ]:
prep.check("J18-B2-Q1", ...)

### Step 2 (Q2): what does "retrieved" mean when no k is given?
Strategy A gives 2 to chocolates with both Cocoa and Hazelnut, else 0. Strategy B gives 2 for both, 1 for exactly one of them, 0 for neither. The question asks how B's precision and recall compare with A's, but **it never says which k**. That gap is the real lesson here, so explore it instead of guessing.

**Step 2a.** On paper, write each chocolate's score under A and under B.

**Step 2b.** Now compare the strategies under two different readings of "retrieved":
- **Reading 1:** a strategy *retrieves* exactly the chocolates it scores above 0. A score of 0 means "no evidence at all", so those are never shown. Compute precision and recall of that set, exactly like notebook 01.
- **Reading 2:** each strategy returns all 9 chocolates as a ranked list (ties by ID). Compare P@k and R@k at every k from 1 to 9.

Write code for both readings in the cell below. You can use `rank_by` and your `precision_recall_at_k`.

In [ ]:
# your attempt here: build score_A and score_B as dicts {chocolate: score}, then compare the two readings

<details><summary>Hint 1</summary>

Under reading 2, the two ranked lists agree at almost every k. At how many values of k do they differ at all? Which of the six options could be "the" answer if the answer depends on a k nobody told you?

</details>

<details><summary>Hint 2</summary>

Under reading 1, Strategy A retrieves only the Cocoa-and-Hazelnut chocolates. Strategy B retrieves those *plus* every chocolate with exactly one of the two. Compute both precisions and both recalls for these two sets.

</details>

<details><summary>Hint 3</summary>

When a question compares "precision and recall" without a k, ask which reading makes the question meaningful. A reading that makes the answer depend on an unstated k can't be what the setter meant. Comparing the *sets each strategy says yes to* is the reading that gives one clear answer.

</details>

In [ ]:
prep.check("J18-B2-Q2", ...)   # option number

### Step 3 (Q3): which DCG property prefers B?
Strategy A returns [D5, D1, D8, D2] and Strategy B returns [D8, D5, D1, D2]. First compute both DCGs (your `dcg` will do the arithmetic once you've written the relevance lists). Then judge each option.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

Which strategy has the higher DCG, and by how much? (It's close.)

</details>

<details><summary>Hint 2</summary>

Option 2 sounds like the textbook description of DCG. But where does each strategy put the most relevant chocolate, D1 (relevance 10)? Does B do *that* better than A?

</details>

<details><summary>Hint 3</summary>

An option is right only if (i) the statement is true of DCG **and** (ii) it explains why B beats A *here*. Test every option against both conditions.

</details>

In [ ]:
prep.check("J18-B2-Q3", ...)   # option number

### Step 4 (Q4): which scoring function gets the best Precision@2?
W = (3, 3, 1, 1). Compute all four scores for all nine chocolates, rank each with ties by ID, and find Precision@2.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

Score A is a weighted sum (a dot product of the ingredient vector with W). Score B just counts how many ingredients the chocolate has, because every weight is at least 1. Score C is Score A minus that count.

</details>

<details><summary>Hint 2</summary>

Score D divides the weighted sum by (1 + number of ingredients). It rewards high weight **per ingredient**, so extra low-weight ingredients drag a chocolate down.

</details>

<details><summary>Hint 3</summary>

Ties matter for the top 2. Sort with the key (−score, ID number), which is what `rank_by` does.

</details>

In [ ]:
prep.check("J18-B2-Q4", ...)   # option number

### Step 5 (Q5): PI-Recall@5
Use the Score A ranking with p = [0.5, 0.5, 0.3, 0.3, 0.2] for positions 1–5. PI-Recall@5 adds p_i only at positions holding a relevant chocolate, then divides by R.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

You already have the Score A ranking from Q4. List its top 5 with their relevances.

</details>

<details><summary>Hint 2</summary>

R is the number of relevant chocolates in the whole collection, the same R as in Q1.

</details>

In [ ]:
prep.check("J18-B2-Q5", ...)

---
# Part 2: Signals

## 7. Convolution, exactly as the exam defines it

Here is the kind of signal from the hook: a slowly swinging arm, sensor noise, and one bump at tick 60.

In [ ]:
t = np.arange(100)
rng = np.random.default_rng(4)
imu = np.sin(t / 12) + rng.normal(0, 0.05, t.size)
imu[60] += 1.0
plt.figure(figsize=(6, 3))
plt.plot(t, imu)
plt.xlabel("tick"); plt.ylabel("IMU reading"); plt.title("Where is the bump?"); plt.grid(alpha=.3)
plt.show()

**Idea.** A **time series** is an ordered list of numbers measured over time: x = [x₁, x₂, …, xₙ]. A **kernel** is a short list h = [h₁, …, h_k]. You slide the kernel along the series, and at each position you multiply and add. The result y is a new series that is large wherever the signal "looks like" the pattern the kernel is built to find.

The Jan 18 paper (page 19) defines it like this:

$$y_i = \sum_{j=1}^{k} h_j \, x_{i+k-j}, \qquad \text{output length } m = n - k + 1 \text{ (no padding)}$$

and spells out three steps: **1. reverse the kernel, 2. slide it along the series, 3. at each position multiply matching entries and add.**

Why "reverse"? Look at the index i + k − j. As j goes 1, 2, …, k, the x-index goes **down**: i+k−1, …, i+1, i. So h₁ multiplies the **last** value in the window and h_k multiplies the **first**. Reversing the kernel lines them up.

**Smallest example.** x = [3, 1, 4, 1, 5] and h = [1, 2, 3].

**PREDICT:** how long is y? What is y₁? (Reverse first!)

In [ ]:
x = [3, 1, 4, 1, 5]
h = [1, 2, 3]
k = len(h)
rev = h[::-1]
print("reversed kernel:", rev)
rows = []
for i in range(len(x) - k + 1):
    window = x[i:i + k]
    products = [a * b for a, b in zip(rev, window)]
    rows.append([i + 1, str(window), " + ".join(f"{a}*{b}" for a, b in zip(rev, window)), sum(products)])
prep.table(rows, header=["i", "window", "reversed kernel x window", "y_i"])
print("formula check, i = 1: h1*x3 + h2*x2 + h3*x1 =", h[0] * x[2] + h[1] * x[1] + h[2] * x[0])

**Padding and stride** (also defined on page 19):
- **Padding p** adds p copies of a value to *each* end of x before sliding. Padding [3, 5, 7] with one 0 on each side gives [0, 3, 5, 7, 0]. The value is usually 0, but a question can choose another one.
- **Stride s** is the step size of the slide. Stride 1 uses every start position. Stride 2 uses positions 1, 3, 5, … (as long as the whole window still fits).

### BUILD 5: `conv_exam`
Return y as a list, using the exam's definition. `pad` is how many copies of `pad_value` go on each end. Start positions are 0, stride, 2·stride, … for as long as a full window fits.

In [ ]:
def conv_exam(x, h, pad=0, pad_value=0, stride=1):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(conv_exam, [
    (([2, 4, 6, 8], [1, -1]), [2, 2, 2]),            # the paper's own example
    (([3, 1, 4, 1, 5], [1, 2, 3]), [15, 12, 19]),
    (([3, 5, 7], [1, 1], 1, 0), [3, 8, 12, 7]),      # padded to [0, 3, 5, 7, 0]
    (([1, 2], [1, 1], 1, 9), [10, 3, 11]),            # padded to [9, 1, 2, 9]
    (([1, 2, 3, 4, 5], [1, 0], 0, 0, 2), [2, 4]),     # stride 2
])

---
## 8. Correlation: what PyTorch actually does

**Idea.** If you've used CNNs, your instinct is to slide the kernel **without** reversing it. That operation is called **correlation**:

$$\text{corr}_i = \sum_{j=1}^{k} h_j \, x_{i+j-1}$$

PyTorch's `Conv1d` and NumPy's `np.correlate` compute correlation, despite the name. NumPy's `np.convolve` reverses the kernel, like the exam. **On the exam, use the paper's definition, not your PyTorch memory** (move **M1**).

### BUILD 6: `correlate`
No reversing, no padding, stride 1.

In [ ]:
def correlate(x, h):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(correlate, [
    (([2, 4, 6, 8], [1, -1]), [-2, -2, -2]),
    (([3, 1, 4, 1, 5], [1, 2, 3]), [17, 12, 21]),
    (([1, 2, 3], [5]), [5, 10, 15]),
])

### EXPLORE
**PREDICT:** on a random signal, which of `np.convolve(x, h, "valid")` and `np.correlate(x, h, "valid")` matches your `conv_exam`, and which matches your `correlate`? And what single change to h turns `correlate` into `conv_exam`?

In [ ]:
if prep.ready(conv_exam, [1, 2], [1]) and prep.ready(correlate, [1, 2], [1]):
    rng = np.random.default_rng(7)
    xr = [int(v) for v in rng.integers(-5, 6, 8)]
    hr = [2, -1, 3]
    print("x =", xr, "  h =", hr)
    prep.table([
        ["conv_exam", conv_exam(xr, hr)],
        ["np.convolve valid", [int(v) for v in np.convolve(xr, hr, "valid")]],
        ["correlate", correlate(xr, hr)],
        ["np.correlate valid", [int(v) for v in np.correlate(xr, hr, "valid")]],
        ["correlate with h reversed", correlate(xr, hr[::-1])],
    ], header=["method", "output"])

**Reversing h turns one into the other.** So:
- for a **symmetric** kernel (reads the same both ways, like [1, −2, 1]) they give identical outputs;
- for an **antisymmetric** kernel (reversing it flips every sign, like [1, 0, −1]) the outputs are exact negatives;
- for anything else they can be quite different. Try h = [1, 2, 3] above.

**Now do practice problem 4.6 on paper**, then check:

In [ ]:
prep.check("4.6_exam", ...)   # the exam's convolution output, as a list
prep.check("4.6_corr", ...)   # the no-flip correlation output, as a list

---
## 9. Kernels are difference operators

**Idea.** Write out what a small kernel actually computes, using the exam's convention y_i = h₁·x_{i+k−1} + … + h_k·x_i.
- h = [1, −1]: y_i = x_{i+1} − x_i. The **change** from one tick to the next. If x is position, this is **velocity**.
- h = [1, −2, 1]: y_i = x_{i+2} − 2x_{i+1} + x_i = (x_{i+2} − x_{i+1}) − (x_{i+1} − x_i). The change **of the change**. If x is position, this is **acceleration**.
- h = [−1, 2, −1]: y_i = (x_{i+1} − x_i) + (x_{i+1} − x_{i+2}). How far the middle value **sticks up above both neighbours**. That's a **spike** detector.

**Smallest example.** A joint moves through positions x = 0, 1, 4, 9, 16, 25 (the squares).

**PREDICT:** what does [1, −1] give? What does [1, −2, 1] give? Work out the first two entries of each by hand.

In [ ]:
if prep.ready(conv_exam, [1, 2], [1]):
    xs = [0, 1, 4, 9, 16, 25]
    print("x            :", xs)
    print("[1, -1]      :", conv_exam(xs, [1, -1]), " <- velocity")
    print("[1, -2, 1]   :", conv_exam(xs, [1, -2, 1]), " <- acceleration")

Constant acceleration, as you'd expect from a curve that bends at a steady rate.

**PREDICT the whole grid** before running: for each signal below, which kernels give all zeros, which give a constant, and which give one big entry?

In [ ]:
signals = {
    "constant": [4, 4, 4, 4, 4, 4],
    "straight line": [1, 3, 5, 7, 9, 11],
    "squares": [0, 1, 4, 9, 16, 25],
    "spike": [1, 1, 1, 6, 1, 1],
}
kernels = {"[1,-1]": [1, -1], "[1,-2,1]": [1, -2, 1], "[-1,2,-1]": [-1, 2, -1]}
if prep.ready(conv_exam, [1, 2], [1]):
    rows = [[name] + [str(conv_exam(sig, h)) for h in kernels.values()] for name, sig in signals.items()]
    prep.table(rows, header=["signal"] + list(kernels))

Two patterns worth naming:
- **Kernels whose entries add to 0 send every constant signal to 0.** If every x is c, then every y_i = c × (h₁ + … + h_k).
- **[1, −2, 1] and [−1, 2, −1] also send every straight line to 0**, because a line has constant velocity, so its velocity never changes.

### EXPLORE: find the bump
**PREDICT:** run the spike detector [−1, 2, −1] over the noisy IMU signal from section 7. Will the bump at tick 60 stand out? Will the noise look bigger or smaller than in the raw signal?

In [ ]:
if prep.ready(conv_exam, [1, 2], [1]):
    y = conv_exam(list(imu), [-1, 2, -1])
    fig, ax = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    ax[0].plot(t, imu); ax[0].set_ylabel("raw"); ax[0].grid(alpha=.3)
    ax[1].plot(t[1:-1], y, color="C3"); ax[1].set_ylabel("[-1, 2, -1]"); ax[1].grid(alpha=.3)
    ax[1].set_xlabel("tick (output i is centred on tick i+1)")
    plt.tight_layout(); plt.show()

The slow swing almost vanishes (it's nearly a straight line over any three ticks), and the bump towers over everything. The noise, though, got *bigger*: differences of noisy numbers are noisier than the numbers. That's why real detectors often smooth first and difference second.

---
## 10. Telescoping: the sum S without computing y

The paper defines two aggregates of the output: **S = y₁ + y₂ + … + y_m** and **S_abs = |y₁| + |y₂| + … + |y_m|**.

**Idea.** With h = [1, −1], each y_i is a difference x_{i+1} − x_i. Add them up and almost everything cancels:

(x₂ − x₁) + (x₃ − x₂) + (x₄ − x₃) + (x₅ − x₄) = x₅ − x₁.

Each middle value appears once with + and once with −. The sum collapses like a folding telescope, so this is called **telescoping**.

**Smallest example.** x = [3, 8, 2, 7, 5].

In [ ]:
x = [3, 8, 2, 7, 5]
terms = [x[i + 1] - x[i] for i in range(len(x) - 1)]
print(" + ".join(f"({x[i + 1]} - {x[i]})" for i in range(len(x) - 1)), "=", " + ".join(map(str, terms)), "=", sum(terms))
print("last - first =", x[-1], "-", x[0], "=", x[-1] - x[0])

**PREDICT (derive it):** for h = [1, −2, 1], find S using only the first two and last two values of x.

<details><summary>Hint 1</summary>

Write d_i = x_{i+1} − x_i for the first differences. Section 9 showed that [1, −2, 1] gives y_i = d_{i+1} − d_i.

</details>

<details><summary>Hint 2</summary>

So S is a sum of differences of the d's, and it telescopes exactly like the example above. Which d survives with a +, and which with a −?

</details>

Run the cell below once you have a formula. It compares S with "(last difference) − (first difference)" on random signals.

In [ ]:
if prep.ready(conv_exam, [1, 2], [1]):
    rng = np.random.default_rng(3)
    rows = []
    for _ in range(4):
        xr = [int(v) for v in rng.integers(0, 10, 7)]
        S = sum(conv_exam(xr, [1, -2, 1]))
        rows.append([str(xr), S, (xr[-1] - xr[-2]) - (xr[1] - xr[0])])
    prep.table(rows, header=["x", "S for [1,-2,1]", "(x_n - x_(n-1)) - (x_2 - x_1)"])

**Aha: S only sees the ends of the signal** (move **M11: kernels telescope**). For [1, −2, 1] it is (last velocity) − (first velocity), whatever happens in between. That's great for questions like "is the signal speeding up overall?" It's useless for "is there a spike somewhere in the middle?", because the middle cancels.

**PREDICT:** for the spike signal [1, 1, 6, 1, 1] and the spike detector [−1, 2, −1], what is S? What is S_abs?

In [ ]:
if prep.ready(conv_exam, [1, 2], [1]):
    y = conv_exam([1, 1, 6, 1, 1], [-1, 2, -1])
    print("y =", y, "  S =", sum(y), "  S_abs =", sum(abs(v) for v in y))

The spike is huge in y, yet S = 0: the positive centre and the two negative shoulders cancel. **S_abs** can't cancel, because every term is ≥ 0. When a question switches from S to S_abs, ask yourself what cancellation it's trying to avoid.

**Now do practice problem 4.5 on paper**, then check:

In [ ]:
prep.check("4.5a", ...)       # y as a list
prep.check("4.5b", ...)       # y as a list
prep.check("4.5b_pos", ...)   # how many entries are positive
prep.check("4.5c", ...)       # S, from the end values only

---
## 11. Designing a kernel from requirements

**Idea.** Exam questions often turn this around: *here are the requirements, which kernel meets them?* Don't guess. Turn each requirement into an **equation on h**, then test the survivors on three probe signals: a constant, a smooth one and a spiky one.

For a length-3 kernel the exam's convention gives y_i = h₁·x_{i+2} + h₂·x_{i+1} + h₃·x_i.
- "**Constant → 0**": a constant c gives y_i = c(h₁ + h₂ + h₃), so you need **h₁ + h₂ + h₃ = 0**.
- "**Straight line → 0**": put x_t = a + b·t. Collect the terms and you get y_i = (a + b·i)(h₁ + h₂ + h₃) + b(2h₁ + h₂). So you need the sum to be 0 **and** 2h₁ + h₂ = 0.

**Worked design.** Requirement: "output the change over **two** ticks, y_i = x_{i+2} − x_i." Match the coefficients of x_{i+2}, x_{i+1}, x_i: h₁ = 1, h₂ = 0, h₃ = −1. So h = [1, 0, −1]. (Correlation thinking would have given [−1, 0, 1], the wrong sign.)

Here is a probe tool that uses **your** `conv_exam`. Give it kernels and signals, and it prints S (or S_abs) for every pair.

**PREDICT** before running it: which of [1, 0, −1] and [1, 1, 1] passes "constant → 0"? For [1, 0, −1], will S be bigger on the smooth signal or on the spiky one? (Telescoping tells you which values of x S depends on.)

In [ ]:
def probe(kernels, signals, agg="S"):
    """Print S (agg="S") or S_abs (agg="S_abs") for every kernel on every signal, using your conv_exam."""
    rows = []
    for kname, h in kernels.items():
        row = [kname]
        for x in signals.values():
            y = conv_exam(x, h)
            row.append(sum(y) if agg == "S" else sum(abs(v) for v in y))
        rows.append(row)
    prep.table(rows, header=[f"{agg}: kernel"] + list(signals))


probe_signals = {"constant": [5, 5, 5, 5, 5], "smooth": [1, 2, 3, 4, 5], "spiky": [1, 5, 1, 5, 1]}
if prep.ready(conv_exam, [1, 2], [1]):
    probe({"[1,0,-1]": [1, 0, -1], "[1,1,1]": [1, 1, 1]}, probe_signals)

[1, 1, 1] fails "constant → 0" straight away, which you could have seen from its sum without any probing. And [1, 0, −1] scores the zig-zag signal 0: its S telescopes to (x₅ + x₄) − (x₂ + x₁), so it measures overall rise and is blind to wiggles in between. Passing one requirement says nothing about the others.

**Your turn (the struggle).** Find **every** length-3 kernel that outputs 0 for **every** straight line. Use the two equations above. How many free choices are left?

<details><summary>Hint 1</summary>

From 2h₁ + h₂ = 0 you get h₂ in terms of h₁. Put that into h₁ + h₂ + h₃ = 0 to get h₃ in terms of h₁.

</details>

<details><summary>Hint 2</summary>

Everything is a multiple of h₁. Pick h₁ = 1 and write the kernel. Which kernel from section 9 is it? What does choosing h₁ = −1 give instead, and how does that change the **sign** of the output?

</details>

Type your kernel below. The check runs it on random straight lines and reports the biggest output. It should be exactly 0.

In [ ]:
my_line_blind = [..., ..., ...]

def check_line_blind(h):
    if any(v is ... for v in h):
        print("Fill in your kernel first."); return
    if not prep.ready(conv_exam, [1, 2], [1]):
        return
    rng = np.random.default_rng(0)
    worst = 0
    for _ in range(20):
        a, b = rng.integers(-9, 10, 2)
        line = [int(a + b * t) for t in range(8)]
        worst = max(worst, max(abs(v) for v in conv_exam(line, h)))
    print(f"largest |y| over 20 random lines: {worst}", "(line-blind)" if worst == 0 else "(not line-blind)")

check_line_blind(my_line_blind)

**Aha: requirements are equations.** "Constant → 0" is one equation, "line → 0" is another, and a length-3 kernel has only 3 numbers. So two requirements pin it down up to a scale factor (and a sign). Use these equations to **eliminate** options fast, then test the few survivors on the specific signals in the question. One failing signal is enough to kill an option (move **M10: kill options with counterexamples**).

---
## 12. Exam walkthrough: Jan 18, Section B3 (Pattern Pursuit: Convolution)

> Do this **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`, pages 19–22.

Do every step **by hand first**, the way you will in the exam. Use `conv_exam` and `probe` only to check.

### Step 1 (Q1): padding with a value, and stride 2
x = [2, 1, 3, 4], h = [1, 2], padding of 1 with value 3 on both ends, stride 2.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

Write the padded series first: one 3 at each end. How long is it, and how many stride-2 windows of length 2 fit?

</details>

<details><summary>Hint 2</summary>

Reverse the kernel before you slide it. The windows start at positions 1, 3, 5, … as long as the whole window fits.

</details>

In [ ]:
prep.check("J18-B3-Q1", ...)   # option letter

### Step 2 (Q2): smooth vs fluctuating
Requirements: constant → S = 0; smooth (like x = [2, 4, 6, 8, 10]) → |S| small; fluctuating (like z = [2, 6, 3, 10, 4]) → |S| large. Eliminate with equations first, then compute S for the survivors.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

"Constant → S = 0" needs h₁ + h₂ + h₃ = 0. Cross out every option that fails it.

</details>

<details><summary>Hint 2</summary>

For each survivor, S telescopes to a few end values (section 10). Compute S on x and on z for each one.

</details>

<details><summary>Hint 3</summary>

x is a straight line. Which survivor gives exactly 0 on *every* straight line, and is its |S| on z large compared with the others?

</details>

In [ ]:
prep.check("J18-B3-Q2", ...)   # option letter

### Step 3 (Q3): acceleration
Requirements: constant velocity → S = 0; accelerating x = [1, 2, 4, 7, 11] → S > 0; decelerating w = [1, 5, 8, 10, 11] → S < 0.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

"Constant velocity" means constant differences, which means a straight line. Use your result from section 11.

</details>

<details><summary>Hint 2</summary>

Now check the **sign** on x. Remember the exam reverses the kernel, and S telescopes to (last difference) − (first difference) for one of the candidate kernels.

</details>

In [ ]:
prep.check("J18-B3-Q3", ...)   # option letter

### Step 4 (Q4): spikes, and why the question switches to S_abs
Requirements on S_abs: constant → 0; smooth (z = [2, 3, 4, 5, 6, 7]) → small; spiky (x = [2, 3, 8, 3, 2, 3]) → large.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

Why S_abs? Take any kernel with entries summing to 0 and compute plain S on the spiky x. Section 10 tells you what to expect.

</details>

<details><summary>Hint 2</summary>

Eliminate with "constant → 0" first. z is a straight line, so "smooth → small" is easiest to satisfy with a kernel that's blind to lines.

</details>

<details><summary>Hint 3</summary>

For the survivors, compute S_abs on x. `probe(..., agg="S_abs")` will check your arithmetic.

</details>

In [ ]:
prep.check("J18-B3-Q4", ...)   # option letter

### Step 5 (Q5): symmetry
A length-4 kernel on a length-4 signal. Symmetric signals → y = 0; asymmetric signals → y ≠ 0.

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

With k = n = 4 there is only one output: y = h₁x₄ + h₂x₃ + h₃x₂ + h₄x₁ (reverse, then multiply and add).

</details>

<details><summary>Hint 2</summary>

Every symmetric length-4 signal looks like [a, b, b, a]. Substitute it and demand y = 0 for **every** a and b. That gives two equations on h.

</details>

<details><summary>Hint 3</summary>

Which options satisfy both equations? Then check that z = [1, 2, 3, −1] gives y ≠ 0.

</details>

In [ ]:
prep.check("J18-B3-Q5", ...)   # option letter

**EXPLORE after you've checked Q5:** find an *asymmetric* length-4 signal that your chosen kernel still maps to 0. (Try signals where two different pairs of entries cancel.) What does that tell you about the word "best" in the question, and about what one number can detect?

In [ ]:
# your exploration here

---
## 13. Connect

Answer in your own words in the cell below. Then copy the best sentence into `log/ERROR_LOG.md` under "Moves I own".

1. In notebook 01, AUC only cared about the **order** of the scores. Which metrics in this notebook are also order-only? If you squared every tf-idf score before ranking, would P@k or DCG change? What if you added 5 to every score?
2. PageRank stops changing when PR = update(PR), a **fixed point**. In notebook 03 (k-means) you also repeated an update until nothing changed, and in notebook 05 you'll shift a recurrence by its fixed point. In each case, what makes the repeated update settle down instead of blowing up? (Think about what d < 1 does to the old values.)
3. A tf-idf score, WonkaRank's Score A and one entry of a convolution are all "multiply matching entries and add": a dot product, s = X·w. For each of the three, what plays the role of the data X and what plays the role of the weights w?

*Your answers here.*